In [70]:
library(tidyr)
library(dplyr)
library(ggplot2)
library(patchwork)
library(data.table)

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/module_projection_fxns.R")

Here I visualize module genes to try to decide on how to clean up the bulk data prior to running FM again (to hopefully boost signal of certain cell types)

In [2]:
options(repr.plot.width=20, repr.plot.height=8, repr.plot.res=150)

In [66]:
mod_def <- "Seed"

## Prep data

### Prep enrichments & bulk data

In [ ]:
# *** Bulk gene expression should be the same dataset used in to find modules and enrichments ***

bulk_data_source <- "GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected"

enrichment_source <- "GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules_MO_2117sets_enrichments"

bulk_expr <- fread(paste0("data/", bulk_data_source, ".csv"), data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"

top_mods_df <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)
top_mods_df$Cell_type <- gsub("/", "_", top_mods_df$Cell_type, fixed=TRUE)
top_mods_df$Cell_type <- gsub(" ", "_", top_mods_df$Cell_type, fixed=TRUE)

In [ ]:
# zero.values <- sum(bulk_expr[,-1] <= 0)

# if(zero.values > 0) {
#     cat("\n")
#     print("Gene expression values <= 0 are present. Scaling all data so minimum expression = 1",quote=F)
#     cat("\n")
#     bulk_expr[,-1] <- bulk_expr[,-1] + abs(min(bulk_expr[,-1], na.rm=T)) + 1
# } 

### Prep single cell data

In [88]:
sc_data_source <- "yao_2021_MOp_STAR_gene_counts_normalized"

sc_expr <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/MOp/yao_2021_MOp_STAR_gene_counts.csv", data.table=FALSE)
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/MOp/yao_2021_MOp_STAR_sampleinfo.csv", data.table=FALSE)

In [89]:
# Remove cell types with only a few cells

min_cells <- 5

ctype_tally <- table(sampleinfo$subclass_label)
ctypes_to_keep <- names(ctype_tally)[ctype_tally >= min_cells]
cells_to_keep <- which(sampleinfo$subclass_label %in% ctypes_to_keep)

sc_expr <- sc_expr[, c(1, cells_to_keep + 1)]
sampleinfo <- sampleinfo[cells_to_keep,]

all.equal(colnames(sc_expr)[-1], sampleinfo$Cell_ID)

[1] TRUE

In [ ]:
colnames(sc_expr)[1] <- "Gene"

In [ ]:
total_expr <- colSums(sc_expr[,-1])
sc_expr_norm <- sweep(sc_expr[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4
sc_expr_norm <- data.frame(Gene=sc_expr[,1], sc_expr_norm) 

### Prep markers

In [79]:
set_source <- "Claude_marker_genes"

human_marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")
mouse_marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_mouse.RDS")

names(marker_genes_list)

[1] "All Neuronal"      "All Glutamatergic" "All GABAergic"    
 [4] "CGE Class"         "MGE Class"         "L2/3 IT"          
 [7] "L4 IT"             "L5 IT"             "L5 ET"            
[10] "L5/6 NP"           "L6 CT"             "L6 IT"            
[13] "L6 IT Car3"        "L6b"               "Lamp5"            
[16] "Lamp5 Lhx6"        "Sncg"              "Vip"              
[19] "Pax6"              "Pvalb"             "Chandelier"       
[22] "Sst"               "Sst Chodl"         "Astro"            
[25] "Oligo"             "OPC"               "Micro/PVM"        
[28] "Endo"              "VLMC"              "Peri"

In [ ]:
# marker_genes_list <- lapply(marker_genes_list, function(x) {
#     if (length(x) > 20) {
#         x[1:20]
#     } else x
# }

## Select modules

In [60]:
head(top_mods_df[,c("Cell_type", "Qval")], 25)

,Cell_type,Qval
,<chr>,<dbl>
1,NAGY_2020_EX1,0.000000e+00
2,MORABITO_2021_EX5_DE_SUBTYPE_CLUSTERS,0.000000e+00
3,AGARWAL_CTX_2020_ASTROCYTE,0.000000e+00
4,CAJIGAS_UP_IN_CA1_NEUROPIL,0.000000e+00
5,SAM_GLIOMA_TURQUOISE_GENES_IDH1_MODULE_6302GENES,0.000000e+00
6,AGARWAL_SN_2020_DANS,0.000000e+00
7,LAU_2020_MICROGLIA,6.926646e-286
8,Layer5_He_NatNeuro_2017,1.643199e-258
9,kelley_microglia,2.259857e-236


In [61]:
top_mods_df_subset <- top_mods_df[1:20,]

## Visualize genes in bulk data

#### Marker genes

In [ ]:
filename <- paste0("figures/", bulk_data_source, "_", set_source, ".pdf")

for (i in seq_along(human_marker_genes_list)) {
    ctype_genes <- human_marker_genes_list[[i]]
    plot_title <- names(human_marker_genes_list)[i] 
    plot_gene_expr_over_samples(bulk_expr, ctype_genes, plot_title, plot_sub=NULL)
}

dev.off()

#### Module genes

In [ ]:
filename <- paste0("figures/", enrichment_source, ".pdf")

max_genes <- 15

for (i in 1:nrow(top_mods_df_subset)) {
    mod <- top_mods_df_subset$Module[i]
    kME_path <- top_mods_df_subset$kME_path[i]
    mod_genes <- get_mod_genes(kME_path, mod, mod_def)
    mod_genes <- na.omit(mod_genes[1:20])

    plot_title <- paste(
        top_mods_df_subset$Cell_type[i], mod_def, 
        "\n", mod, top_mods_df_subset$Network[i]
    ) 
    plot_sub <- paste("Qval:", round(top_mods_df_subset$Qval[i], 4))

    plot_gene_expr_over_samples(bulk_expr, mod_genes, plot_title, plot_sub, target_species=NULL)
}

## Visualize genes in single cell data

### Marker genes

In [ ]:
ctype_assignment_vec <- sampleinfo$subclass_label

for (i in seq_along(mouse_marker_genes_list)) {
    gene_vec <- mouse_marker_genes_list[[i]]
    plot_title <- names(mouse_marker_genes_list)[i]
    plot_sub <- NULL

    plot_gene_projections(sc_expr_norm, gene_vec, ctype_assignment_vec, plot_title, plot_sub)
}

### Module genes

In [ ]:
for (i in 1:nrow(top_mods_df_subset)) {
    mod <- top_mods_df_subset$Module[i]
    kME_path <- top_mods_df_subset$kME_path[i]
    gene_vec <- get_mod_genes(kME_path, mod, mod_def)

    plot_title <- paste(top_mods_df_subset$Cell_type[i], mod_def, "\n", mod, top_mods_df$Network[i]) 
    plot_sub <- paste("Qval:", round(top_mods_df_subset$Qval[i], 4))

    plot_gene_projections(sc_expr_norm, gene_vec, ctype_assignment_vec, plot_title, plot_sub, target_species="mouse")
}